# ChromaDB storing and retrieval

In [1]:
# pip install chromadb, sentence-transformers

In [1]:
# Importing the necessary modules from the chromadb package:
# chromadb is used to interact with the Chroma DB database,
# embedding_functions is used to define the embedding model
import chromadb
from chromadb.utils import embedding_functions

# Define the embedding function using SentenceTransformers
ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# Create a new instance of ChromaClient to interact with the Chroma DB
client = chromadb.Client()

# Define the name for the collection to be created or retrieved
collection_name = "my_grocery_collection"

### Create a collection

In [2]:
# Create a collection in the Chroma database with a specified name, 
# distance metric, and embedding function. In this case, we are using 
# cosine distance
collection = client.create_collection(
    name=collection_name,
    metadata={"description": "A collection for storing grocery data"},
    configuration={
        "hnsw": {"space": "cosine"}, # Default space is l2 (Euclidean). Other valid options are cosine and ip (inner product / dot product)
        "embedding_function": ef
    }
)
print(f"Collection created: {collection.name}")

Collection created: my_grocery_collection


In [3]:
# Array of grocery-related text items
texts = [
    'fresh red apples',
    'organic bananas',
    'ripe mangoes',
    'whole wheat bread',
    'farm-fresh eggs',
    'natural yogurt',
    'frozen vegetables',
    'grass-fed beef',
    'free-range chicken',
    'fresh salmon fillet',
    'aromatic coffee beans',
    'pure honey',
    'golden apple',
    'red fruit'
]

# Create a list of unique IDs for each text item in the 'texts' array
# Each ID follows the format 'food_<index>', where <index> starts from 1
ids = [f"food_{index + 1}" for index, _ in enumerate(texts)]

### Add documents and metadatas to collections

In [4]:
# Add documents and their corresponding IDs to the collection
# The `add` method inserts or update the data into the collection
# The documents are the actual text items, and the IDs are unique identifiers
# ChromaDB will automatically generate embeddings using the configured embedding function
collection.add(
    documents=texts,
    metadatas=[{"source": "grocery_store", "category": "food"} for _ in texts],
    ids=ids
)

In [ ]:
# Retrieve all the items (documents) stored in the collection
all_items = collection.get()

print("Collection contents:")
print(f"Number of documents: {len(all_items['documents'])}")

Collection contents:
Number of documents: 14


In [7]:
all_items['documents']

['fresh red apples',
 'organic bananas',
 'ripe mangoes',
 'whole wheat bread',
 'farm-fresh eggs',
 'natural yogurt',
 'frozen vegetables',
 'grass-fed beef',
 'free-range chicken',
 'fresh salmon fillet',
 'aromatic coffee beans',
 'pure honey',
 'golden apple',
 'red fruit']

In [7]:
all_items.keys()

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas'])

### Update an item in a collection

In [5]:
collection.update(
    ids=['food_1'],
    metadatas=[{"source": "my_grocery_store", "new_category": "food"}], # update value of existing "source" key and add new key of "new_category"
    documents=['very fresh red apples']
)

In [6]:
all_items = collection.get()
print(all_items['documents'][0])
print(all_items['metadatas'][0])

very fresh red apples
{'category': 'food', 'source': 'my_grocery_store', 'new_category': 'food'}


### Delete an item in a collection

In [ ]:
# collection.delete(
#     ids=['food_1'],
#     where={"new_category": "food"}
# )

### Get reference of collection from client

In [22]:
collection_again = client.get_collection(name=collection_name)
collection_again.get(ids=['food_1'])['ids']

['food_1']

## Perform similarity search in the collection

In [8]:
 # Define the query term you want to search for in the collection
query_term = "apple"

# Perform a query to search for the most similar documents to the 'query_term'
results = collection.query(
    query_texts=[query_term],
    n_results=3  # Retrieve top 3 results
)
print(f"Query results for '{query_term}':")
results

Query results for 'apple':


{'ids': [['food_13', 'food_1', 'food_14']],
 'embeddings': None,
 'documents': [['golden apple', 'very fresh red apples', 'red fruit']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'category': 'food', 'source': 'grocery_store'},
   {'new_category': 'food', 'source': 'my_grocery_store', 'category': 'food'},
   {'source': 'grocery_store', 'category': 'food'}]],
 'distances': [[0.3824648857116699, 0.5658824443817139, 0.5965151786804199]]}

In [9]:
results['ids'][0]

['food_13', 'food_1', 'food_14']

In [ ]:
print(f'Top 3 similar documents to "{query_term}":')

# Access the nested arrays in 'results["ids"]' and 'results["distances"]'
result_ids = results['ids'][0]
result_distances = results['distances'][0]
result_documents = results['documents'][0]
for i in range(min(3, len(result_ids))):
    doc_id = result_ids[i]  # Get ID from 'ids' array
    score = result_distances[i]  # Get score from 'distances' array
    similarity_score = 1 - result_distances # Cosine Similarity Score = 1 - distance
    
    text = result_documents[i] # Retrieve text data from the results
    if not text:
        print(f' - ID: {doc_id}, Text: "Text not available", Score: {score:.4f}')
    else:
        print(f' - ID: {doc_id}, Text: "{text}", Score: {score:.4f}')

Top 3 similar documents to "apple":
 - ID: food_13, Text: "golden apple", Score: 0.3825
 - ID: food_1, Text: "very fresh red apples", Score: 0.5659
 - ID: food_14, Text: "red fruit", Score: 0.5965


In [11]:
# multiple queries in one call
query_terms = ["apple", "honey"]

# Perform a query to search for the most similar documents to the 'query_term'
results = collection.query(
    query_texts=query_terms,
    n_results=3,  # Retrieve top 3 results
    # where={"category": "food"} # exact match from metadata. Its similar to SQL where clause
    # where_document={"$contains": "apple"} # exact match of text from documents. Its similar to like query in SQL. Its also known as full-text search
)
print(f"Query results for '{' and '.join(query_terms)}':")
results

Query results for 'apple and honey':


{'ids': [['food_13', 'food_1', 'food_14'], ['food_12', 'food_14', 'food_13']],
 'embeddings': None,
 'documents': [['golden apple', 'very fresh red apples', 'red fruit'],
  ['pure honey', 'red fruit', 'golden apple']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'source': 'grocery_store', 'category': 'food'},
   {'category': 'food', 'new_category': 'food', 'source': 'my_grocery_store'},
   {'category': 'food', 'source': 'grocery_store'}],
  [{'category': 'food', 'source': 'grocery_store'},
   {'category': 'food', 'source': 'grocery_store'},
   {'category': 'food', 'source': 'grocery_store'}]],
 'distances': [[0.3824650049209595, 0.5658824443817139, 0.5965152382850647],
  [0.24796128273010254, 0.689453125, 0.7210021615028381]]}

In [12]:
for index, query_term in enumerate(query_terms):
    print(f'Top 3 similar documents to "{query_term}":')

    for i, (doc_id, document, distance) in enumerate(zip(results['ids'][0], results['documents'][0], results['distances'][0])):
        print(f' - ID: {doc_id}, Text: "{document}", Score: {distance:.4f}')

Top 3 similar documents to "apple":
 - ID: food_13, Text: "golden apple", Score: 0.3825
 - ID: food_1, Text: "very fresh red apples", Score: 0.5659
 - ID: food_14, Text: "red fruit", Score: 0.5965
Top 3 similar documents to "honey":
 - ID: food_13, Text: "golden apple", Score: 0.3825
 - ID: food_1, Text: "very fresh red apples", Score: 0.5659
 - ID: food_14, Text: "red fruit", Score: 0.5965


## Metadata Filtering Operators

**Basic equality**\
where={"key": "value"}

**Equivalent to**\
where={"key": {"$eq": "value"}}

**Comparison operators**\
"$eq"   # equal to (string, int, float)\
"$ne"   # not equal to\
"$gt"   # greater than (int, float)\
"$gte"  # greater than or equal to\
"$lt"   # less than (int, float)\
"$lte"  # less than or equal to\
"$in"   # in list\
"$nin"  # not in list

```
# Using a list to perform an OR operation on the values of a metadata key
collection.get(
    where={
        "$and": [
            {"source": {"$in": ["langchain.com", "llamaindex.ai"]}}, 
            {"version": {"$lt": 0.3}}
        ]
    }
)
```

## Document Content Filtering

**Contains text**\
where_document={"$contains": "pandas"}

**Does not contain text**\
where_document={"$not_contains": "library"}

**Combined with logical operators**\
```
where_document={
    "$or": [
        {"$contains": "LangChain"},
        {"$contains": "Python"}
    ]
}
```

---

## Advance Example

In [15]:
from  employees_record import employees
employees[0]

{'id': 'employee_1',
 'name': 'John Doe',
 'experience': 5,
 'department': 'Engineering',
 'role': 'Software Engineer',
 'skills': 'Python, JavaScript, React, Node.js, databases',
 'location': 'New York',
 'employment_type': 'Full-time'}

In [13]:
collection_name2 = "employee_collection"

# Create another collection
collection2 = client.create_collection(
    name=collection_name2,
    metadata={"description": "A collection for storing employee data"},
    configuration={
        "hnsw": {"space": "cosine"},
        "embedding_function": ef
    }
)
print(f"Collection created: {collection.name}")

Collection created: my_grocery_collection


In [16]:
# Create comprehensive text documents for each employee
# These documents will be used for similarity search based on skills, roles, and experience
employee_documents = []
for employee in employees:
    document = f"{employee['role']} with {employee['experience']} years of experience in {employee['department']}. "
    document += f"Skills: {employee['skills']}. Located in {employee['location']}. "
    document += f"Employment type: {employee['employment_type']}."
    employee_documents.append(document)

In [17]:
# Adding data to the collection in the Chroma database
collection2.add(
    ids=[employee["id"] for employee in employees],
    documents=employee_documents,
    metadatas=[{
        "name": employee["name"],
        "department": employee["department"],
        "role": employee["role"],
        "experience": employee["experience"],
        "location": employee["location"],
        "employment_type": employee["employment_type"]
    } for employee in employees]
)

In [18]:
print("=== Combined Search: Similarity + Metadata Filtering ===")

print("Finding senior Python developers in major tech cities:")
query_text = "senior Python developer full-stack"
results = collection2.query(
    query_texts=[query_text],
    n_results=5,
    where={
        "$and": [
            {"experience": {"$gte": 8}},
            {"location": {"$in": ["San Francisco", "New York", "Seattle"]}}
        ]
    }
)

# results

=== Combined Search: Similarity + Metadata Filtering ===
Finding senior Python developers in major tech cities:


In [19]:
print(f"Query: '{query_text}' with filters (8+ years, major tech cities)")
print(f"Found {len(results['ids'][0])} matching employees:")
for i, (doc_id, document, distance) in enumerate(zip(
    results['ids'][0], results['documents'][0], results['distances'][0]
)):
    metadata = results['metadatas'][0][i]
    print(f"  {i+1}. {metadata['name']} ({doc_id}) - Distance: {distance:.4f}")
    print(f"     {metadata['role']} in {metadata['location']} ({metadata['experience']} years)")
    print(f"     Document snippet: {document[:80]}...")

Query: 'senior Python developer full-stack' with filters (8+ years, major tech cities)
Found 4 matching employees:
  1. Michael Brown (employee_4) - Distance: 0.6726
     Senior Software Engineer in San Francisco (12 years)
     Document snippet: Senior Software Engineer with 12 years of experience in Engineering. Skills: Jav...
  2. Chris Evans (employee_8) - Distance: 0.7537
     Senior Architect in New York (20 years)
     Document snippet: Senior Architect with 20 years of experience in Engineering. Skills: System desi...
  3. David Lee (employee_6) - Distance: 0.8344
     Engineering Manager in Seattle (15 years)
     Document snippet: Engineering Manager with 15 years of experience in Engineering. Skills: Team lea...
  4. Olivia Moore (employee_15) - Distance: 0.8761
     Principal Engineer in San Francisco (12 years)
     Document snippet: Principal Engineer with 12 years of experience in Engineering. Skills: Technical...
